In [1]:
import pandas as pd
import warnings
from IPython.utils import io
import sys
import numpy as np
from functools import reduce

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

stars_dir = '~/GitHub/stars-data-builder/'
hos_dir = '~/Desktop/Rush/CMS_HospitalArchives/'

dates_df = pd.DataFrame(columns = ['Measure ID', 'Start Date', 'End Date'])

In [2]:
def curate(df):

    try:
        df = df[df['PROVIDER_ID'] != np.nan]
        df['PROVIDER_ID'] = df['PROVIDER_ID'].values.astype(str)
        
        ids = df['PROVIDER_ID'].tolist()
        ids2 = []
        for i in ids:
            if len(i) < 6:
                i = '0' + i
            ids2.append(i)
        df['PROVIDER_ID'] = ids2
        
    except:
        pass
    
    for c in list(df):    
        try:
            df[c] = df[c].str.replace("\t","")
        except:
            pass

    if 'Unnamed: 0' in list(df):
        df.drop(labels=['Unnamed: 0'], axis=1, inplace=True)
    return df

# Use 2026 R input file to get the column labels needed for 2027 predictions


In [3]:
sas_input_df = pd.read_csv(stars_dir + '2026/2026-05_Stars_Release/stars_alldata_2026apr.csv')
sas_input_df.head()

,PROVIDER_ID,COMP_HIP_KNEE_DEN,EDAC_30_AMI,EDAC_30_AMI_DEN,EDAC_30_HF,EDAC_30_HF_DEN,EDAC_30_PN,EDAC_30_PN_DEN,HAI_1_DEN_VOL,HAI_1_DEN_PRED,HAI_1,HAI_2_DEN_VOL,HAI_2_DEN_PRED,HAI_2,HAI_3_DEN_VOL,HAI_3_DEN_PRED,HAI_3,HAI_4_DEN_VOL,HAI_4_DEN_PRED,HAI_4,HAI_5_DEN_VOL,HAI_5_DEN_PRED,HAI_5,HAI_6_DEN_VOL,HAI_6_DEN_PRED,HAI_6,H_CLEAN_LINEAR_SCORE,H_COMP_1_LINEAR_SCORE,H_COMP_2_LINEAR_SCORE,H_COMP_3_LINEAR_SCORE,H_COMP_5_LINEAR_SCORE,H_COMP_6_LINEAR_SCORE,H_COMP_7_LINEAR_SCORE,H_HSP_RATING_LINEAR_SCORE,H_NUMB_COMP,H_QUIET_LINEAR_SCORE,H_RECMND_LINEAR_SCORE,H_RESP_RATE_P,Hybrid_HWM_DEN,Hybrid_HWM_RSMR,Hybrid_HWR_DEN,Hybrid_HWR,IMM_3,IMM_3_DEN,MORT_30_AMI_DEN,MORT_30_CABG_DEN,MORT_30_COPD_DEN,MORT_30_HF_DEN,MORT_30_PN_DEN,MORT_30_STK_DEN,OP_10_DEN,OP_10,OP_13_DEN,OP_13,OP_18B_DEN,OP_18B,OP_22_DEN,OP_22,OP_23_DEN,OP_23,OP_29_DEN,OP_29,OP_32_DEN,OP_35_ADM_DEN,OP_35_ED_DEN,OP_36_DEN,OP-37_NUMB_COMP,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,OP_8_DEN,OP_8,PSI_4_SURG_COMP_DEN,PSI_4_SURG_COMP,PSI_90_SAFETY_DEN,PSI_90_SAFETY,READM_30_CABG_DEN,READM_30_COPD_DEN,READM_30_HIP_KNEE_DEN,COMP_HIP_KNEE,OP_32,OP_35_ADM,OP_35_ED,OP_36,MORT_30_AMI,MORT_30_CABG,MORT_30_COPD,MORT_30_HF,MORT_30_PN,MORT_30_STK,READM_30_CABG,READM_30_COPD,READM_30_HIP_KNEE,SAFE_USE_OF_OPIOIDS_DEN,SAFE_USE_OF_OPIOIDS,SEP_1_DEN,SEP_1
0,010001,27.0,-15.6,273.0,-1.1,652.0,17.4,507.0,8935.0,9.440,0.530,16255.0,23.350,0.086,229.0,6.562,0.457,NaN,NaN,NaN,109019.0,10.937,0.183,109019.0,68.076,0.382,85.0,90.0,92.0,82.0,79.0,87.0,82.0,89.0,627.0,88.0,91.0,17.0,1835.0,0.045,2824.0,0.151,0.93,4625.0,270.0,144.0,112.0,583.0,517.0,395.0,1517.0,0.053,365.0,0.038,387.0,222.0,57084.0,0.05,11.0,0.91,29.0,0.72,218.0,261.0,261.0,647.0,185.0,98.0,93.0,97.0,94.0,94.0,52.0,0.308,118.0,203.00,NaN,0.95,137.0,122.0,25.0,0.032,12.8,8.5,5.7,0.8,0.114,0.030,0.094,0.102,0.184,0.135,0.101,0.180,0.048,4583.0,0.14,150.0,0.69
1,010005,104.0,NaN,NaN,12.2,164.0,-17.2,292.0,4514.0,2.746,0.728,6567.0,2.991,1.003,90.0,2.291,0.436,NaN,NaN,NaN,37163.0,1.258,1.590,35488.0,9.436,0.530,85.0,91.0,93.0,77.0,72.0,88.0,80.0,87.0,586.0,87.0,85.0,14.0,698.0,0.046,986.0,0.133,0.59,2856.0,NaN,NaN,126.0,158.0,285.0,89.0,828.0,0.128,153.0,0.033,1120.0,137.0,58624.0,0.03,11.0,0.27,210.0,1.00,897.0,99.0,99.0,396.0,752.0,98.0,96.0,98.0,94.0,93.0,83.0,0.422,27.0,184.79,NaN,0.97,NaN,132.0,81.0,0.030,14.2,8.6,6.2,1.0,NaN,NaN,0.089,0.141,0.212,0.129,NaN,0.171,0.042,1859.0,0.15,289.0,0.75
2,010006,49.0,-18.0,285.0,-4.7,461.0,-0.9,679.0,3915.0,4.335,0.000,7561.0,9.758,0.000,92.0,2.448,0.000,225.0,2.183,0.0,69033.0,4.733,0.000,66038.0,38.671,0.078,79.0,89.0,88.0,76.0,70.0,82.0,77.0,83.0,1724.0,84.0,80.0,17.0,1583.0,0.052,2494.0,0.159,0.64,2565.0,266.0,79.0,160.0,413.0,659.0,258.0,898.0,0.088,221.0,0.041,336.0,150.0,44924.0,0.01,15.0,0.67,85.0,0.86,1618.0,NaN,NaN,502.0,326.0,98.0,94.0,97.0,93.0,91.0,NaN,NaN,91.0,236.12,NaN,1.14,71.0,174.0,56.0,0.047,11.6,NaN,NaN,1.0,0.145,0.054,0.087,0.125,0.196,0.124,0.106,0.191,0.051,4350.0,0.15,137.0,0.66
3,010007,NaN,NaN,NaN,77.9,39.0,29.8,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4132.0,1.481,0.675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,125.0,0.048,189.0,0.154,0.29,358.0,NaN,NaN,34.0,34.0,98.0,NaN,135.0,0.059,NaN,NaN,1107.0,117.0,12667.0,0.02,NaN,NaN,52.0,0.65,117.0,NaN,NaN,NaN,180.0,97.0,92.0,96.0,91.0,92.0,NaN,NaN,NaN,NaN,NaN,1.06,NaN,32.0,NaN,NaN,12.5,NaN,NaN,NaN,NaN,NaN,0.112,0.134,0.254,NaN,NaN,0.186,NaN,212.0,0.15,15.0,0.13
4,010008,NaN,NaN,NaN,NaN,NaN,-8.9,29.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50.0,0.043,88.0,0.147,0.31,110.0,NaN,NaN,NaN,NaN,31.0,NaN,54.0,0.019,NaN,NaN,350.0,112.0,6062.0,0.01,NaN,NaN,12.0,0.67,60.0,NaN,NaN,NaN,61.0,98.0,96.0,97.0,93.0,93.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.150,NaN,NaN,NaN,NaN,65.0,0.18,NaN,NaN


In [4]:
#sas_input_df = pd.read_sas(stars_dir + '2025/2025-07 Stars Release/alldata_2025jul.sas7bdat', 
#                           format = 'sas7bdat', encoding = "utf8")

sas_input_df = pd.read_csv(stars_dir + '2026/2026-05_Stars_Release/stars_alldata_2026apr.csv')

ls = ['PC_01', 
      'PC_01_DEN',
      'READM_30_HOSP_WIDE', 
      'READM_30_HOSP_WIDE_DEN',
      'H_NUMB_COMP', 
      'H_RESP_RATE_P',
      'HCP_COVID_19',
      'HCP_COVID_19_DEN',
      'H_COMP_1_STAR_RATING', 
      'H_COMP_2_STAR_RATING', 
      'H_COMP_3_STAR_RATING', 
      'H_COMP_5_STAR_RATING', 
      'H_COMP_6_STAR_RATING', 
      'H_COMP_7_STAR_RATING',
      'H_GLOB_STAR_RATING', 
      'H_INDI_STAR_RATING', 
      'OP-37_NUMB_COMP',
     ]
for l in ls:
    try:
        sas_input_df.drop(labels=[l], axis=1, inplace=True)
    except:
        print(l, 'not present')
        pass
    
ls = list(sas_input_df)
for l in ls:
    if '_DEN' in l:
        print('dropping', l)
        sas_input_df.drop(labels=[l], axis=1, inplace=True)
        
sas_cols_2026 = list(sas_input_df)

ls = ['Hybrid_HWR',
      #'PC_02', #'PC_05', 
      #'ePC_07a', #'ePC_07b',
      #'HH-01', 'HH-02',
      'Hybrid_HWM',
      'O-COMP-1',
      'O-COMP-2',
      'O-COMP-3',
      'O-PATIENT-RATE',
      'O-PATIENT-REC',
      'H_COMP_1_LINEAR_SCORE',
      'H_COMP_2_LINEAR_SCORE', 
      'H_COMP_3_LINEAR_SCORE', 
      'H_COMP_5_LINEAR_SCORE', 
      'H_COMP_6_LINEAR_SCORE', 
      'H_COMP_7_LINEAR_SCORE', 
      'H_CLEAN_LINEAR_SCORE',  
      'H_QUIET_LINEAR_SCORE', 
      'H_RECMND_LINEAR_SCORE', 
      'H_HSP_RATING_LINEAR_SCORE',
     ]

for l in ls:
    sas_cols_2026.append(l)

sas_cols_2026 = list(set(sas_cols_2026))    
sas_cols = list(sas_cols_2026)

sas_input_df = curate(sas_input_df)

print()
print(len(sas_cols_2026), ' features')
print(len(sas_cols_2026) - 1, ' measures:', sorted(sas_cols_2026), '\n')

sas_input_df.head()

PC_01 not present
PC_01_DEN not present
READM_30_HOSP_WIDE not present
READM_30_HOSP_WIDE_DEN not present
HCP_COVID_19 not present
HCP_COVID_19_DEN not present
H_COMP_1_STAR_RATING not present
H_COMP_2_STAR_RATING not present
H_COMP_3_STAR_RATING not present
H_COMP_5_STAR_RATING not present
H_COMP_6_STAR_RATING not present
H_COMP_7_STAR_RATING not present
H_GLOB_STAR_RATING not present
H_INDI_STAR_RATING not present
dropping COMP_HIP_KNEE_DEN
dropping EDAC_30_AMI_DEN
dropping EDAC_30_HF_DEN
dropping EDAC_30_PN_DEN
dropping HAI_1_DEN_VOL
dropping HAI_1_DEN_PRED
dropping HAI_2_DEN_VOL
dropping HAI_2_DEN_PRED
dropping HAI_3_DEN_VOL
dropping HAI_3_DEN_PRED
dropping HAI_4_DEN_VOL
dropping HAI_4_DEN_PRED
dropping HAI_5_DEN_VOL
dropping HAI_5_DEN_PRED
dropping HAI_6_DEN_VOL
dropping HAI_6_DEN_PRED
dropping Hybrid_HWM_DEN
dropping Hybrid_HWR_DEN
dropping IMM_3_DEN
dropping MORT_30_AMI_DEN
dropping MORT_30_CABG_DEN
dropping MORT_30_COPD_DEN
dropping MORT_30_HF_DEN
dropping MORT_30_PN_DEN
droppi

,PROVIDER_ID,EDAC_30_AMI,EDAC_30_HF,EDAC_30_PN,HAI_1,HAI_2,HAI_3,HAI_4,HAI_5,HAI_6,H_CLEAN_LINEAR_SCORE,H_COMP_1_LINEAR_SCORE,H_COMP_2_LINEAR_SCORE,H_COMP_3_LINEAR_SCORE,H_COMP_5_LINEAR_SCORE,H_COMP_6_LINEAR_SCORE,H_COMP_7_LINEAR_SCORE,H_HSP_RATING_LINEAR_SCORE,H_QUIET_LINEAR_SCORE,H_RECMND_LINEAR_SCORE,Hybrid_HWM_RSMR,Hybrid_HWR,IMM_3,OP_10,OP_13,OP_18B,OP_22,OP_23,OP_29,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,OP_8,PSI_4_SURG_COMP,PSI_90_SAFETY,COMP_HIP_KNEE,OP_32,OP_35_ADM,OP_35_ED,OP_36,MORT_30_AMI,MORT_30_CABG,MORT_30_COPD,MORT_30_HF,MORT_30_PN,MORT_30_STK,READM_30_CABG,READM_30_COPD,READM_30_HIP_KNEE,SAFE_USE_OF_OPIOIDS,SEP_1
0,010001,-15.6,-1.1,17.4,0.530,0.086,0.457,NaN,0.183,0.382,85.0,90.0,92.0,82.0,79.0,87.0,82.0,89.0,88.0,91.0,0.045,0.151,0.93,0.053,0.038,222.0,0.05,0.91,0.72,98.0,93.0,97.0,94.0,94.0,0.308,203.00,0.95,0.032,12.8,8.5,5.7,0.8,0.114,0.030,0.094,0.102,0.184,0.135,0.101,0.180,0.048,0.14,0.69
1,010005,NaN,12.2,-17.2,0.728,1.003,0.436,NaN,1.590,0.530,85.0,91.0,93.0,77.0,72.0,88.0,80.0,87.0,87.0,85.0,0.046,0.133,0.59,0.128,0.033,137.0,0.03,0.27,1.00,98.0,96.0,98.0,94.0,93.0,0.422,184.79,0.97,0.030,14.2,8.6,6.2,1.0,NaN,NaN,0.089,0.141,0.212,0.129,NaN,0.171,0.042,0.15,0.75
2,010006,-18.0,-4.7,-0.9,0.000,0.000,0.000,0.0,0.000,0.078,79.0,89.0,88.0,76.0,70.0,82.0,77.0,83.0,84.0,80.0,0.052,0.159,0.64,0.088,0.041,150.0,0.01,0.67,0.86,98.0,94.0,97.0,93.0,91.0,NaN,236.12,1.14,0.047,11.6,NaN,NaN,1.0,0.145,0.054,0.087,0.125,0.196,0.124,0.106,0.191,0.051,0.15,0.66
3,010007,NaN,77.9,29.8,NaN,NaN,NaN,NaN,NaN,0.675,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.048,0.154,0.29,0.059,NaN,117.0,0.02,NaN,0.65,97.0,92.0,96.0,91.0,92.0,NaN,NaN,1.06,NaN,12.5,NaN,NaN,NaN,NaN,NaN,0.112,0.134,0.254,NaN,NaN,0.186,NaN,0.15,0.13
4,010008,NaN,NaN,-8.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.043,0.147,0.31,0.019,NaN,112.0,0.01,NaN,0.67,98.0,96.0,97.0,93.0,93.0,NaN,NaN,NaN,NaN,12.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.150,NaN,NaN,NaN,NaN,0.18,NaN


## HAIs

In [5]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/Healthcare_Associated_Infections-Hospital.csv')
#print(df['Measure ID'].unique())

measures = ['HAI_1_ELIGCASES', 'HAI_1_DOPC', 'HAI_1_SIR', 'HAI_2_ELIGCASES', 'HAI_2_DOPC', 'HAI_2_SIR', 
            'HAI_3_ELIGCASES', 'HAI_3_DOPC', 'HAI_3_SIR', 'HAI_4_ELIGCASES', 'HAI_4_DOPC', 'HAI_4_SIR', 
            'HAI_5_ELIGCASES', 'HAI_5_DOPC', 'HAI_5_SIR', 'HAI_6_ELIGCASES', 'HAI_6_DOPC', 'HAI_6_SIR']

tdf = df[df['Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0)

df = df[df['Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'Measure ID', 'Score'], axis=1)

hai_df = pd.DataFrame(columns=['Facility ID']) 
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    hai_df = hai_df.merge(tdf2, on='Facility ID', how='outer')
    
hai_df.rename(columns={'HAI_1_ELIGCASES': 'HAI_1_DEN_PRED',
                       'HAI_1_DOPC': 'HAI_1_DEN_VOL',
                       'HAI_1_SIR': 'HAI_1',
                       'HAI_2_ELIGCASES': 'HAI_2_DEN_PRED',
                       'HAI_2_DOPC': 'HAI_2_DEN_VOL',
                       'HAI_2_SIR': 'HAI_2',
                       'HAI_3_ELIGCASES': 'HAI_3_DEN_PRED',
                       'HAI_3_DOPC': 'HAI_3_DEN_VOL',
                       'HAI_3_SIR': 'HAI_3',
                       'HAI_4_ELIGCASES': 'HAI_4_DEN_PRED',
                       'HAI_4_DOPC': 'HAI_4_DEN_VOL',
                       'HAI_4_SIR': 'HAI_4',
                       'HAI_5_ELIGCASES': 'HAI_5_DEN_PRED',
                       'HAI_5_DOPC': 'HAI_5_DEN_VOL',
                       'HAI_5_SIR': 'HAI_5',
                       'HAI_6_ELIGCASES': 'HAI_6_DEN_PRED',
                       'HAI_6_DOPC': 'HAI_6_DEN_VOL',
                       'HAI_6_SIR': 'HAI_6',
                       'Facility ID': 'PROVIDER_ID',
                   }, inplace=True)

for c in list(hai_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
hai_df = curate(hai_df)

47 remaining features: ['COMP_HIP_KNEE', 'EDAC_30_AMI', 'EDAC_30_HF', 'EDAC_30_PN', 'H_CLEAN_LINEAR_SCORE', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_3_LINEAR_SCORE', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'H_HSP_RATING_LINEAR_SCORE', 'H_QUIET_LINEAR_SCORE', 'H_RECMND_LINEAR_SCORE', 'Hybrid_HWM', 'Hybrid_HWM_RSMR', 'Hybrid_HWR', 'IMM_3', 'MORT_30_AMI', 'MORT_30_CABG', 'MORT_30_COPD', 'MORT_30_HF', 'MORT_30_PN', 'MORT_30_STK', 'O-COMP-1', 'O-COMP-2', 'O-COMP-3', 'O-PATIENT-RATE', 'O-PATIENT-REC', 'OP_10', 'OP_13', 'OP_18B', 'OP_22', 'OP_23', 'OP_29', 'OP_32', 'OP_35_ADM', 'OP_35_ED', 'OP_36', 'OP_8', 'PSI_4_SURG_COMP', 'PSI_90_SAFETY', 'READM_30_CABG', 'READM_30_COPD', 'READM_30_HIP_KNEE', 'SAFE_USE_OF_OPIOIDS', 'SEP_1'] 



## Unplanned Hospital Visits


In [6]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/Unplanned_Hospital_Visits-Hospital.csv')
measures = ['EDAC_30_AMI', 'EDAC_30_HF', 'EDAC_30_PN', 'OP_32', 'OP_35_ADM', 'OP_35_ED', 'OP_36', 
            'READM_30_CABG', 'READM_30_COPD', 'READM_30_HIP_KNEE', 'Hybrid_HWR']

tdf = df[df['Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0)

df = df[df['Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'Denominator', 'Measure ID', 'Score'], axis=1)

uhv_df = pd.DataFrame(columns=['Facility ID'])
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    tdf2[m + '_DEN'] = tdf1['Denominator'].tolist()
    uhv_df = uhv_df.merge(tdf2, on='Facility ID', how='outer')

uhv_df.rename(columns={'Facility ID': 'PROVIDER_ID'}, inplace=True)

for c in list(uhv_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass
    
print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
uhv_df = curate(uhv_df)

36 remaining features: ['COMP_HIP_KNEE', 'H_CLEAN_LINEAR_SCORE', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_3_LINEAR_SCORE', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'H_HSP_RATING_LINEAR_SCORE', 'H_QUIET_LINEAR_SCORE', 'H_RECMND_LINEAR_SCORE', 'Hybrid_HWM', 'Hybrid_HWM_RSMR', 'IMM_3', 'MORT_30_AMI', 'MORT_30_CABG', 'MORT_30_COPD', 'MORT_30_HF', 'MORT_30_PN', 'MORT_30_STK', 'O-COMP-1', 'O-COMP-2', 'O-COMP-3', 'O-PATIENT-RATE', 'O-PATIENT-REC', 'OP_10', 'OP_13', 'OP_18B', 'OP_22', 'OP_23', 'OP_29', 'OP_8', 'PSI_4_SURG_COMP', 'PSI_90_SAFETY', 'SAFE_USE_OF_OPIOIDS', 'SEP_1'] 



## COMPLICATIONS AND DEATHS

In [7]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/Complications_and_Deaths-Hospital.csv')

measures = ['MORT_30_AMI', 'MORT_30_CABG', 'MORT_30_COPD', 'MORT_30_HF', 
            'MORT_30_PN', 'MORT_30_STK', 'PSI_04', 'COMP_HIP_KNEE',
            'PSI_90', 
            'Hybrid_HWM',
           ]

tdf = df[df['Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0, ignore_index=True)


df = df[df['Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'Measure ID', 'Score', 'Denominator'], axis=1)

cad_df = pd.DataFrame(columns=['Facility ID'])
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    
    tdf2[m + '_DEN'] = tdf1['Denominator'].tolist()
    cad_df = cad_df.merge(tdf2, on='Facility ID', how='outer')
    
cad_df.rename(columns={'Facility ID': 'PROVIDER_ID',
                       'PSI_04': 'PSI_4_SURG_COMP',
                       'PSI_04_DEN': 'PSI_4_SURG_COMP_DEN',
                       'PSI_90': 'PSI_90_SAFETY',
                       'PSI_90_DEN': 'PSI_90_SAFETY_DEN',
                   }, inplace=True)

for c in list(cad_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass
    
print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
cad_df = curate(cad_df)

26 remaining features: ['H_CLEAN_LINEAR_SCORE', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_3_LINEAR_SCORE', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'H_HSP_RATING_LINEAR_SCORE', 'H_QUIET_LINEAR_SCORE', 'H_RECMND_LINEAR_SCORE', 'Hybrid_HWM_RSMR', 'IMM_3', 'O-COMP-1', 'O-COMP-2', 'O-COMP-3', 'O-PATIENT-RATE', 'O-PATIENT-REC', 'OP_10', 'OP_13', 'OP_18B', 'OP_22', 'OP_23', 'OP_29', 'OP_8', 'SAFE_USE_OF_OPIOIDS', 'SEP_1'] 



## TIMELY AND EFFECTIVE CARE

Everything except PC-01, which for 2024 is located in the Maternal Health files of the hospitals data archive

In [8]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/Timely_and_Effective_Care-Hospital.csv')

measures = ['IMM_3', 'OP_18b', 'OP_22', 'OP_23', 'OP_29', 'SEP_1',
            'SAFE_USE_OF_OPIOIDS', 
           ]

tdf = df[df['Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0, ignore_index=True)


df = df[df['Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'Sample', 'Measure ID', 'Score'], axis=1)

tec_df = pd.DataFrame(columns=['Facility ID'])
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    
    if m == 'HCP_COVID_19':
        pass
    else:
        tdf2[m + '_DEN'] = tdf1['Sample'].tolist()
    
    tec_df = tec_df.merge(tdf2, on='Facility ID', how='outer')
    
tec_df.rename(columns={'Facility ID': 'PROVIDER_ID', 
                       'OP_18b': 'OP_18B',
                       'OP_18b_DEN': 'OP_18B_DEN',
                      }, inplace=True)

for c in list(tec_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
tec_df = curate(tec_df)

df_dupes = tec_df[tec_df.duplicated('PROVIDER_ID', keep=False)]

print(df_dupes.shape)
df_dupes.head()

19 remaining features: ['H_CLEAN_LINEAR_SCORE', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_3_LINEAR_SCORE', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'H_HSP_RATING_LINEAR_SCORE', 'H_QUIET_LINEAR_SCORE', 'H_RECMND_LINEAR_SCORE', 'Hybrid_HWM_RSMR', 'O-COMP-1', 'O-COMP-2', 'O-COMP-3', 'O-PATIENT-RATE', 'O-PATIENT-REC', 'OP_10', 'OP_13', 'OP_8'] 

(0, 15)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN


In [9]:
def collapse_hospitals(df, id_col='PROVIDER_ID'):
    """
    Collapse multiple rows per hospital into a single row by
    taking the first non-null value in each column.
    """

    def first_non_null(series):
        non_null = series.dropna()
        return non_null.iloc[0] if not non_null.empty else np.nan

    collapsed_df = (
        df
        .groupby(id_col, as_index=False)
        .agg(first_non_null)
    )

    return collapsed_df


# --- Apply to your dataframe ---
tec_df = collapse_hospitals(tec_df)

df_dupes = tec_df[tec_df.duplicated('PROVIDER_ID', keep=False)]

print(df_dupes.shape)
df_dupes.head()

(0, 15)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN


In [10]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/VA_TE.csv')

measures = ['IMM-3', 'OP-18b', 'OP-22', 'OP-23', 'OP-29', 'SEP-1', #'HCP-COVID-19',
           ]
# Does not have 'SAFE_USE_OF_OPIOIDS', 'HH-01', 'HH-02'

tec3_df = pd.DataFrame(columns=['Facility ID'])
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    
    tdf2[m + '_DEN'] = tdf1['Sample'].tolist()
    
    tec3_df = tec3_df.merge(tdf2, on='Facility ID', how='outer')
    
tec3_df.rename(columns={'Facility ID': 'PROVIDER_ID', 
                        'OP-18b': 'OP_18B',
                        'OP-18b_DEN': 'OP_18B_DEN',
                        
                        'IMM-3': 'IMM_3', 
                        'IMM-3_DEN': 'IMM_3_DEN', 
                        
                        'OP-22': 'OP_22', 
                        'OP-22_DEN': 'OP_22_DEN', 
                        
                        'OP-23': 'OP_23', 
                        'OP-23_DEN': 'OP_23_DEN', 
                        
                        'OP-29': 'OP_29', 
                        'OP-29_DEN': 'OP_29_DEN', 
                        
                        'SEP-1': 'SEP_1', 
                        'SEP-1_DEN': 'SEP_1_DEN', 
                        
                        #'HCP-COVID-19': 'HCP_COVID_19',
                        #'HCP-COVID-19_DEN': 'HCP_COVID_19_DEN',
                        
                      }, inplace=True)

tec3_df.replace('Not Available', np.nan, inplace=True)
print(tec3_df.shape)
tec3_df = curate(tec3_df)
print(tec3_df.shape)

print(tec_df.shape)
print(tec3_df.shape)
tec_df = pd.concat([tec_df, tec3_df])
print(tec_df.shape)

tec_df.head()

(132, 13)
(132, 13)
(4660, 15)
(132, 13)
(4792, 15)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN
0,010001,93,4625,217,395,3,62163,Not Available,Not Available,62,39,65,141,14,4583
1,010005,59,2856,141,1165,3,60663,Not Available,Not Available,98,301,73,293,15,1859
2,010006,64,2565,144,320,1,46884,82,17,84,67,61,137,15,4350
3,010007,29,358,128,1048,1,11011,Not Available,Not Available,87,136,20,15,15,212
4,010011,79,2317,156,254,Not Available,Not Available,64,11,Not Available,Not Available,72,129,13,727


## TIMELY AND EFFECTIVE CARE

Note: Maternal Health measures (e.g., PC-02, PC-07a, PC-07b) were not included in the Nov 2025 update due to widespread reporting errors

## HCAHPS

In [11]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/HCAHPS-Hospital.csv')
#df = pd.read_csv(stars_dir + 'CareCompare/hospitals_11_2025/HCAHPS-Hospital.csv')

measures = df['HCAHPS Measure ID'].unique().tolist()

print(len(measures))
print(len(list(set(measures))))

for m in measures:
    print('--'+m+'--')
    
#df.head()

68
68
--H_COMP_1_A_P--
--H_COMP_1_SN_P--
--H_COMP_1_U_P--
--H_COMP_1_LINEAR_SCORE--
--H_COMP_1_STAR_RATING--
--H_NURSE_RESPECT_A_P--
--H_NURSE_RESPECT_SN_P--
--H_NURSE_RESPECT_U_P--
--H_NURSE_LISTEN_A_P--
--H_NURSE_LISTEN_SN_P--
--H_NURSE_LISTEN_U_P--
--H_NURSE_EXPLAIN_A_P--
--H_NURSE_EXPLAIN_SN_P--
--H_NURSE_EXPLAIN_U_P--
--H_COMP_2_A_P--
--H_COMP_2_SN_P--
--H_COMP_2_U_P--
--H_COMP_2_LINEAR_SCORE--
--H_COMP_2_STAR_RATING--
--H_DOCTOR_RESPECT_A_P--
--H_DOCTOR_RESPECT_SN_P--
--H_DOCTOR_RESPECT_U_P--
--H_DOCTOR_LISTEN_A_P--
--H_DOCTOR_LISTEN_SN_P--
--H_DOCTOR_LISTEN_U_P--
--H_DOCTOR_EXPLAIN_A_P--
--H_DOCTOR_EXPLAIN_SN_P--
--H_DOCTOR_EXPLAIN_U_P--
--H_COMP_5_A_P--
--H_COMP_5_SN_P--
--H_COMP_5_U_P--
--H_COMP_5_LINEAR_SCORE--
--H_COMP_5_STAR_RATING--
--H_MED_FOR_A_P--
--H_MED_FOR_SN_P--
--H_MED_FOR_U_P--
--H_SIDE_EFFECTS_A_P--
--H_SIDE_EFFECTS_SN_P--
--H_SIDE_EFFECTS_U_P--
--H_COMP_6_N_P--
--H_COMP_6_Y_P--
--H_COMP_6_LINEAR_SCORE--
--H_COMP_6_STAR_RATING--
--H_DISCH_HELP_N_P--
--H_DISCH_HEL

In [12]:
tdf = df[df['HCAHPS Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf.rename(columns={'HCAHPS Measure ID': 'Measure ID'}, inplace=True)
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0, ignore_index=True)

df = df[df['HCAHPS Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'HCAHPS Measure ID', 'HCAHPS Linear Mean Value', 
                        'Number of Completed Surveys', 'Survey Response Rate Percent'], axis=1)

HCAHPS_df = pd.DataFrame(columns=['Facility ID'])
for i, m in enumerate(measures):
    if m in list(df['HCAHPS Measure ID'].unique().tolist()):
        tdf1 = df[df['HCAHPS Measure ID'] == m]
        tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
        tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
        tdf2[m] = tdf1['HCAHPS Linear Mean Value'].tolist()
        #if i == 0:
        #    tdf2['H_NUMB_COMP'] = tdf1['Number of Completed Surveys'].tolist()
        #    tdf2['H_RESP_RATE_P'] = tdf1['Survey Response Rate Percent'].tolist()

        HCAHPS_df = HCAHPS_df.merge(tdf2, on='Facility ID', how='outer')
    

HCAHPS_df.rename(columns={'Facility ID': 'PROVIDER_ID'}, inplace=True)

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')

for c in list(set(list(HCAHPS_df))):
    try:
        print('removing', c)
        sas_cols_2026.remove(c)
    except:
        print('     FAILED')
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
HCAHPS_df = curate(HCAHPS_df)

HCAHPS_df.head()

19 remaining features: ['H_CLEAN_LINEAR_SCORE', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_3_LINEAR_SCORE', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'H_HSP_RATING_LINEAR_SCORE', 'H_QUIET_LINEAR_SCORE', 'H_RECMND_LINEAR_SCORE', 'Hybrid_HWM_RSMR', 'O-COMP-1', 'O-COMP-2', 'O-COMP-3', 'O-PATIENT-RATE', 'O-PATIENT-REC', 'OP_10', 'OP_13', 'OP_8'] 

removing H_NURSE_EXPLAIN_A_P
     FAILED
removing H_NURSE_LISTEN_U_P
     FAILED
removing H_QUIET_LINEAR_SCORE
removing H_DOCTOR_RESPECT_A_P
     FAILED
removing H_DOCTOR_EXPLAIN_SN_P
     FAILED
removing H_COMP_5_LINEAR_SCORE
removing H_QUIET_HSP_SN_P
     FAILED
removing H_NURSE_RESPECT_A_P
     FAILED
removing H_MED_FOR_SN_P
     FAILED
removing H_SYMPTOMS_N_P
     FAILED
removing H_COMP_2_SN_P
     FAILED
removing H_CLEAN_HSP_A_P
     FAILED
removing H_NURSE_LISTEN_A_P
     FAILED
removing H_RECMND_PY
     FAILED
removing H_COMP_1_LINEAR_SCORE
removing H_COMP_6_N_P
     FAILED
removing H_STAR_RA

,PROVIDER_ID,H_COMP_1_A_P,H_COMP_1_SN_P,H_COMP_1_U_P,H_COMP_1_LINEAR_SCORE,H_COMP_1_STAR_RATING,H_NURSE_RESPECT_A_P,H_NURSE_RESPECT_SN_P,H_NURSE_RESPECT_U_P,H_NURSE_LISTEN_A_P,H_NURSE_LISTEN_SN_P,H_NURSE_LISTEN_U_P,H_NURSE_EXPLAIN_A_P,H_NURSE_EXPLAIN_SN_P,H_NURSE_EXPLAIN_U_P,H_COMP_2_A_P,H_COMP_2_SN_P,H_COMP_2_U_P,H_COMP_2_LINEAR_SCORE,H_COMP_2_STAR_RATING,H_DOCTOR_RESPECT_A_P,H_DOCTOR_RESPECT_SN_P,H_DOCTOR_RESPECT_U_P,H_DOCTOR_LISTEN_A_P,H_DOCTOR_LISTEN_SN_P,H_DOCTOR_LISTEN_U_P,H_DOCTOR_EXPLAIN_A_P,H_DOCTOR_EXPLAIN_SN_P,H_DOCTOR_EXPLAIN_U_P,H_COMP_5_A_P,H_COMP_5_SN_P,H_COMP_5_U_P,H_COMP_5_LINEAR_SCORE,H_COMP_5_STAR_RATING,H_MED_FOR_A_P,H_MED_FOR_SN_P,H_MED_FOR_U_P,H_SIDE_EFFECTS_A_P,H_SIDE_EFFECTS_SN_P,H_SIDE_EFFECTS_U_P,H_COMP_6_N_P,H_COMP_6_Y_P,H_COMP_6_LINEAR_SCORE,H_COMP_6_STAR_RATING,H_DISCH_HELP_N_P,H_DISCH_HELP_Y_P,H_SYMPTOMS_N_P,H_SYMPTOMS_Y_P,H_CLEAN_HSP_A_P,H_CLEAN_HSP_SN_P,H_CLEAN_HSP_U_P,H_CLEAN_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_QUIET_HSP_A_P,H_QUIET_HSP_SN_P,H_QUIET_HSP_U_P,H_QUIET_LINEAR_SCORE,H_QUIET_STAR_RATING,H_HSP_RATING_0_6,H_HSP_RATING_7_8,H_HSP_RATING_9_10,H_HSP_RATING_LINEAR_SCORE,H_HSP_RATING_STAR_RATING,H_RECMND_DN,H_RECMND_DY,H_RECMND_PY,H_RECMND_LINEAR_SCORE,H_RECMND_STAR_RATING,H_STAR_RATING
0,010001,Not Applicable,Not Applicable,Not Applicable,90,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,92,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,78,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,86,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,85,Not Applicable,Not Applicable,Not Applicable,Not Applicable,87,Not Applicable,Not Applicable,Not Applicable,Not Applicable,89,Not Applicable,Not Applicable,Not Applicable,Not Applicable,91,Not Applicable,Not Applicable
1,010005,Not Applicable,Not Applicable,Not Applicable,90,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,92,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,72,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,85,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,83,Not Applicable,Not Applicable,Not Applicable,Not Applicable,86,Not Applicable,Not Applicable,Not Applicable,Not Applicable,86,Not Applicable,Not Applicable,Not Applicable,Not Applicable,84,Not Applicable,Not Applicable
2,010006,Not Applicable,Not Applicable,Not Applicable,89,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,87,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,70,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,82,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,81,Not Applicable,Not Applicable,Not Applicable,Not Applicable,84,Not Applicable,Not Applicable,Not Applicable,Not Applicable,83,Not Applicable,Not Applicable,Not Applicable,Not Appl

#
# OCAHPS

In [13]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/OQR_OAS_CAHPS_BY_HOSPITAL.csv')
measures = df['OAS CAHPS Measure ID'].unique().tolist()

rd = {
    'O_COMP_1_LINEAR_SCORE': 'O-COMP-1',
    'O_COMP_2_LINEAR_SCORE': 'O-COMP-2',
    'O_COMP_3_LINEAR_SCORE': 'O-COMP-3',
    'O_PATIENT_RATE_LINEAR_SCORE': 'O-PATIENT-RATE',
    'O_PATIENT_REC_LINEAR_SCORE': 'O-PATIENT-REC',
}

df['OAS CAHPS Measure ID'].replace(rd, inplace=True)

for m in measures:
    print(m)
    
df.head()

O_COMP_1_LINEAR_SCORE
O_COMP_1_N_P
O_COMP_1_YD_P
O_COMP_1_YS_P
O_COMP_2_LINEAR_SCORE
O_COMP_2_N_P
O_COMP_2_YD_P
O_COMP_2_YS_P
O_COMP_3_LINEAR_SCORE
O_COMP_3_N_P
O_COMP_3_YD_P
O_COMP_3_YS_P
O_PATIENT_RATE_0_6_P
O_PATIENT_RATE_7_8_P
O_PATIENT_RATE_9_10_P
O_PATIENT_RATE_LINEAR_SCORE
O_PATIENT_REC_LINEAR_SCORE
O_PATIENT_REC_NPD_P
O_PATIENT_REC_YD_P
O_PATIENT_REC_YP_P


,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,OAS CAHPS Measure ID,OAS CAHPS Question,OAS CAHPS Answer Description,OAS CAHPS Answer Percent,OAS CAHPS Answer Percent Footnote,OAS CAHPS Linear Mean Value,Number of Completed Surveys,Number of Completed Surveys Footnote,Survey Response Rate Percent,Survey Response Rate Percent Footnote,Start Date,End Date
0,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,O-COMP-1,Facilities and Staff - linear mean score,Facilities and Staff - linear mean score,Not Applicable,NaN,98,456,NaN,17,NaN,07/01/2024,06/30/2025
1,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,O_COMP_1_N_P,"Patients who reported that the facility was ""n...","Facility was ""not"" clean and staff were ""not"" ...",0,NaN,Not Applicable,456,NaN,17,NaN,07/01/2024,06/30/2025
2,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,O_COMP_1_YD_P,"Patients who reported that ""definitely"" the fa...","Facility was ""definitely"" clean and staff were...",96,NaN,Not Applicable,456,NaN,17,NaN,07/01/2024,06/30/2025
3,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,O_COMP_1_YS_P,"Patients who reported ""somewhat"" that the faci...","Facility was ""somewhat"" clean and staff were ""...",4,NaN,Not Applicable,456,NaN,17,NaN,07/01/2024,06/30/2025
4,10001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,O-COMP-2,Communications about your procedure - linear m...,Communications about your procedure - linear m...,Not Applicable,NaN,92,456,NaN,17,NaN,07/01/2024,06/30/2025


In [14]:
measures = [
    'O-COMP-1',
    'O-COMP-2',
    'O-COMP-3',
    'O-PATIENT-RATE',
    'O-PATIENT-REC',
]

tdf = df[df['OAS CAHPS Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf.rename(columns={'OAS CAHPS Measure ID': 'Measure ID'}, inplace=True)
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0, ignore_index=True)


df = df[df['OAS CAHPS Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'OAS CAHPS Measure ID', 'OAS CAHPS Linear Mean Value',
                        'Number of Completed Surveys', 'Survey Response Rate Percent'], axis=1)

OAS_CAHPS_df = pd.DataFrame(columns=['Facility ID'])
for i, m in enumerate(measures):
    tdf1 = df[df['OAS CAHPS Measure ID'] == m]
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['OAS CAHPS Linear Mean Value'].tolist()
        
    OAS_CAHPS_df = OAS_CAHPS_df.merge(tdf2, on='Facility ID', how='outer')

OAS_CAHPS_df.rename(columns={'Facility ID': 'PROVIDER_ID'}, inplace=True)

for c in list(OAS_CAHPS_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
OAS_CAHPS_df = curate(OAS_CAHPS_df)

6 remaining features: ['H_COMP_3_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'Hybrid_HWM_RSMR', 'OP_10', 'OP_13', 'OP_8'] 



In [15]:
measures = [
    'O-COMP-1',
    'O-COMP-2',
    'O-COMP-3',
    'O-PATIENT-RATE',
    'O-PATIENT-REC',
]

for l in measures:
    ls = pd.to_numeric(OAS_CAHPS_df[l], errors='coerce')
    print(np.nanmin(ls), np.nanmedian(ls), np.nanmean(ls), np.nanmax(ls))

80.0 98.0 98.17221693625119 100.0
80.0 95.0 95.28068506184586 100.0
87.0 98.0 97.5359974627339 100.0
69.0 94.0 94.04440215667618 98.0
64.0 93.0 92.73897875039644 99.0


## Outpatient Imaging Efficiency


In [16]:
df = pd.read_csv(stars_dir + 'CareCompare/hospitals_05_2026/Outpatient_Imaging_Efficiency-Hospital.csv')
measures = ['OP-13', 'OP-8', 'OP-10']

tdf = df[df['Measure ID'].isin(measures + ['End Date', 'Start Date'])]
tdf = tdf.filter(items = ['Measure ID', 'Start Date', 'End Date'])
tdf.drop_duplicates(inplace=True)
dates_df = pd.concat([dates_df, tdf], axis=0, ignore_index=True)


df = df[df['Measure ID'].isin(measures)]
df = df.filter(items = ['Facility ID', 'Measure ID', 'Score'], axis=1)

oie_df = pd.DataFrame(columns=['Facility ID'])
for m in measures:
    tdf1 = df[df['Measure ID'] == m]
    
    tdf2 = pd.DataFrame(columns=['Facility ID', m]) 
    tdf2['Facility ID'] = tdf1['Facility ID'].tolist()
    tdf2[m] = tdf1['Score'].tolist()
    
    oie_df = oie_df.merge(tdf2, on='Facility ID', how='outer')
    
    
oie_df.rename(columns={'Facility ID': 'PROVIDER_ID',
                       'OP-13': 'OP_13',
                       'OP-8': 'OP_8',
                       'OP-10': 'OP_10',
                   }, inplace=True)

for c in list(oie_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
oie_df1 = curate(oie_df)


3 remaining features: ['H_COMP_3_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'Hybrid_HWM_RSMR'] 



## MERGE DATAFRAME AND COMPARE TO SAS FILE

In [17]:
print(dates_df.shape)
dates_df.to_csv(stars_dir + "2024/measure_dates/11_2025_measure_dates.csv", index=False)
dates_df.head()

(122, 3)


,Measure ID,Start Date,End Date
0,HAI_1_DOPC,07/01/2024,06/30/2025
1,HAI_1_ELIGCASES,07/01/2024,06/30/2025
2,HAI_1_SIR,07/01/2024,06/30/2025
3,HAI_2_DOPC,07/01/2024,06/30/2025
4,HAI_2_ELIGCASES,07/01/2024,06/30/2025


In [18]:
main_df = tec_df.merge(cad_df, on='PROVIDER_ID', how='outer')
main_df = main_df.merge(HCAHPS_df, on='PROVIDER_ID', how='outer')
main_df = main_df.merge(OAS_CAHPS_df, on='PROVIDER_ID', how='outer')
main_df = main_df.merge(uhv_df, on='PROVIDER_ID', how='outer')
main_df = main_df.merge(hai_df, on='PROVIDER_ID', how='outer')
main_df = main_df.merge(oie_df1, on='PROVIDER_ID', how='outer')

for c in list(main_df):
    try:
        sas_cols_2026.remove(c)
    except:
        pass

print(len(sas_cols_2026), 'remaining features:', sorted(sas_cols_2026), '\n')
print(main_df.shape)
main_df.head()

3 remaining features: ['H_COMP_3_LINEAR_SCORE', 'H_COMP_7_LINEAR_SCORE', 'Hybrid_HWM_RSMR'] 

(4792, 151)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN,MORT_30_AMI,MORT_30_AMI_DEN,MORT_30_CABG,MORT_30_CABG_DEN,MORT_30_COPD,MORT_30_COPD_DEN,MORT_30_HF,MORT_30_HF_DEN,MORT_30_PN,MORT_30_PN_DEN,MORT_30_STK,MORT_30_STK_DEN,PSI_4_SURG_COMP,PSI_4_SURG_COMP_DEN,COMP_HIP_KNEE,COMP_HIP_KNEE_DEN,PSI_90_SAFETY,PSI_90_SAFETY_DEN,Hybrid_HWM,Hybrid_HWM_DEN,H_COMP_1_A_P,H_COMP_1_SN_P,H_COMP_1_U_P,H_COMP_1_LINEAR_SCORE,H_COMP_1_STAR_RATING,H_NURSE_RESPECT_A_P,H_NURSE_RESPECT_SN_P,H_NURSE_RESPECT_U_P,H_NURSE_LISTEN_A_P,H_NURSE_LISTEN_SN_P,H_NURSE_LISTEN_U_P,H_NURSE_EXPLAIN_A_P,H_NURSE_EXPLAIN_SN_P,H_NURSE_EXPLAIN_U_P,H_COMP_2_A_P,H_COMP_2_SN_P,H_COMP_2_U_P,H_COMP_2_LINEAR_SCORE,H_COMP_2_STAR_RATING,H_DOCTOR_RESPECT_A_P,H_DOCTOR_RESPECT_SN_P,H_DOCTOR_RESPECT_U_P,H_DOCTOR_LISTEN_A_P,H_DOCTOR_LISTEN_SN_P,H_DOCTOR_LISTEN_U_P,H_DOCTOR_EXPLAIN_A_P,H_DOCTOR_EXPLAIN_SN_P,H_DOCTOR_EXPLAIN_U_P,H_COMP_5_A_P,H_COMP_5_SN_P,H_COMP_5_U_P,H_COMP_5_LINEAR_SCORE,H_COMP_5_STAR_RATING,H_MED_FOR_A_P,H_MED_FOR_SN_P,H_MED_FOR_U_P,H_SIDE_EFFECTS_A_P,H_SIDE_EFFECTS_SN_P,H_SIDE_EFFECTS_U_P,H_COMP_6_N_P,H_COMP_6_Y_P,H_COMP_6_LINEAR_SCORE,H_COMP_6_STAR_RATING,H_DISCH_HELP_N_P,H_DISCH_HELP_Y_P,H_SYMPTOMS_N_P,H_SYMPTOMS_Y_P,H_CLEAN_HSP_A_P,H_CLEAN_HSP_SN_P,H_CLEAN_HSP_U_P,H_CLEAN_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_QUIET_HSP_A_P,H_QUIET_HSP_SN_P,H_QUIET_HSP_U_P,H_QUIET_LINEAR_SCORE,H_QUIET_STAR_RATING,H_HSP_RATING_0_6,H_HSP_RATING_7_8,H_HSP_RATING_9_10,H_HSP_RATING_LINEAR_SCORE,H_HSP_RATING_STAR_RATING,H_RECMND_DN,H_RECMND_DY,H_RECMND_PY,H_RECMND_LINEAR_SCORE,H_RECMND_STAR_RATING,H_STAR_RATING,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,EDAC_30_AMI,EDAC_30_AMI_DEN,EDAC_30_HF,EDAC_30_HF_DEN,EDAC_30_PN,EDAC_30_PN_DEN,OP_32,OP_32_DEN,OP_35_ADM,OP_35_ADM_DEN,OP_35_ED,OP_35_ED_DEN,OP_36,OP_36_DEN,READM_30_CABG,READM_30_CABG_DEN,READM_30_COPD,READM_30_COPD_DEN,READM_30_HIP_KNEE,READM_30_HIP_KNEE_DEN,Hybrid_HWR,Hybrid_HWR_DEN,HAI_1_DEN_PRED,HAI_1_DEN_VOL,HAI_1,HAI_2_DEN_PRED,HAI_2_DEN_VOL,HAI_2,HAI_3_DEN_PRED,HAI_3_DEN_VOL,HAI_3,HAI_4_DEN_PRED,HAI_4_DEN_VOL,HAI_4,HAI_5_DEN_PRED,HAI_5_DEN_VOL,HAI_5,HAI_6_DEN_PRED,HAI_6_DEN_VOL,HAI_6,OP_13,OP_8,OP_10
0,010001,93,4625,217,395,3,62163,Not Available,Not Available,62,39,65,141,14,4583,11.4,270,3,144,9.4,112,10.2,583,18.4,517,13.5,395,203.00,118,3.2,27,0.95,Not Applicable,4.5,1835,Not Applicable,Not Applicable,Not Applicable,90,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,92,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,78,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,86,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,85,Not Applicable,Not Applicable,Not Applicable,Not Applicable,87,Not Applicable,Not Applicable,Not Applicable,Not Applicable,89,Not Applicable,Not Applicable,Not Applicable,Not Applicable,91,Not Applicable,Not Applicable,98,92,97,94,94,-15.6,273,-1.1,652,17.4,507,12.7,234,10,302,5.1,302,1.1,651,10.1,137,18,122,4.8,25,15.1,2824,10.399,9871,0.288,25.693,18164,0.195,6.830,238,0.293,0.871,91,Not Available,9.613,114863,0.416,71.953,113805,0.459,3.8,30.8,5.3
1,010005,59,2856,141,1165,3,60663,Not Available,Not Available,98,301,73,293,15,1859,Not Available,Not Available,Not Available,Not Available,8.9,126,14.1,158,21.2,285,12.9,89,184.79,27,3,104,0.97,Not Applicable,4.6,698,Not Applicable,Not Applicable,Not Applicable,90,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicab

In [19]:
df_dupes = main_df[main_df.duplicated('PROVIDER_ID', keep=False)]

print(df_dupes.shape)
df_dupes.head()

(0, 151)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN,MORT_30_AMI,MORT_30_AMI_DEN,MORT_30_CABG,MORT_30_CABG_DEN,MORT_30_COPD,MORT_30_COPD_DEN,MORT_30_HF,MORT_30_HF_DEN,MORT_30_PN,MORT_30_PN_DEN,MORT_30_STK,MORT_30_STK_DEN,PSI_4_SURG_COMP,PSI_4_SURG_COMP_DEN,COMP_HIP_KNEE,COMP_HIP_KNEE_DEN,PSI_90_SAFETY,PSI_90_SAFETY_DEN,Hybrid_HWM,Hybrid_HWM_DEN,H_COMP_1_A_P,H_COMP_1_SN_P,H_COMP_1_U_P,H_COMP_1_LINEAR_SCORE,H_COMP_1_STAR_RATING,H_NURSE_RESPECT_A_P,H_NURSE_RESPECT_SN_P,H_NURSE_RESPECT_U_P,H_NURSE_LISTEN_A_P,H_NURSE_LISTEN_SN_P,H_NURSE_LISTEN_U_P,H_NURSE_EXPLAIN_A_P,H_NURSE_EXPLAIN_SN_P,H_NURSE_EXPLAIN_U_P,H_COMP_2_A_P,H_COMP_2_SN_P,H_COMP_2_U_P,H_COMP_2_LINEAR_SCORE,H_COMP_2_STAR_RATING,H_DOCTOR_RESPECT_A_P,H_DOCTOR_RESPECT_SN_P,H_DOCTOR_RESPECT_U_P,H_DOCTOR_LISTEN_A_P,H_DOCTOR_LISTEN_SN_P,H_DOCTOR_LISTEN_U_P,H_DOCTOR_EXPLAIN_A_P,H_DOCTOR_EXPLAIN_SN_P,H_DOCTOR_EXPLAIN_U_P,H_COMP_5_A_P,H_COMP_5_SN_P,H_COMP_5_U_P,H_COMP_5_LINEAR_SCORE,H_COMP_5_STAR_RATING,H_MED_FOR_A_P,H_MED_FOR_SN_P,H_MED_FOR_U_P,H_SIDE_EFFECTS_A_P,H_SIDE_EFFECTS_SN_P,H_SIDE_EFFECTS_U_P,H_COMP_6_N_P,H_COMP_6_Y_P,H_COMP_6_LINEAR_SCORE,H_COMP_6_STAR_RATING,H_DISCH_HELP_N_P,H_DISCH_HELP_Y_P,H_SYMPTOMS_N_P,H_SYMPTOMS_Y_P,H_CLEAN_HSP_A_P,H_CLEAN_HSP_SN_P,H_CLEAN_HSP_U_P,H_CLEAN_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_QUIET_HSP_A_P,H_QUIET_HSP_SN_P,H_QUIET_HSP_U_P,H_QUIET_LINEAR_SCORE,H_QUIET_STAR_RATING,H_HSP_RATING_0_6,H_HSP_RATING_7_8,H_HSP_RATING_9_10,H_HSP_RATING_LINEAR_SCORE,H_HSP_RATING_STAR_RATING,H_RECMND_DN,H_RECMND_DY,H_RECMND_PY,H_RECMND_LINEAR_SCORE,H_RECMND_STAR_RATING,H_STAR_RATING,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,EDAC_30_AMI,EDAC_30_AMI_DEN,EDAC_30_HF,EDAC_30_HF_DEN,EDAC_30_PN,EDAC_30_PN_DEN,OP_32,OP_32_DEN,OP_35_ADM,OP_35_ADM_DEN,OP_35_ED,OP_35_ED_DEN,OP_36,OP_36_DEN,READM_30_CABG,READM_30_CABG_DEN,READM_30_COPD,READM_30_COPD_DEN,READM_30_HIP_KNEE,READM_30_HIP_KNEE_DEN,Hybrid_HWR,Hybrid_HWR_DEN,HAI_1_DEN_PRED,HAI_1_DEN_VOL,HAI_1,HAI_2_DEN_PRED,HAI_2_DEN_VOL,HAI_2,HAI_3_DEN_PRED,HAI_3_DEN_VOL,HAI_3,HAI_4_DEN_PRED,HAI_4_DEN_VOL,HAI_4,HAI_5_DEN_PRED,HAI_5_DEN_VOL,HAI_5,HAI_6_DEN_PRED,HAI_6_DEN_VOL,HAI_6,OP_13,OP_8,OP_10


In [20]:

cols = []
for col in list(main_df):
    if 'PROVIDER_ID' in col:
        continue
    elif '_DEN' in col:
        continue
    else:
        cols.append(col)
        
print(sorted(cols), '\n')
print(len(cols), 'measures')

['COMP_HIP_KNEE', 'EDAC_30_AMI', 'EDAC_30_HF', 'EDAC_30_PN', 'HAI_1', 'HAI_2', 'HAI_3', 'HAI_4', 'HAI_5', 'HAI_6', 'H_CLEAN_HSP_A_P', 'H_CLEAN_HSP_SN_P', 'H_CLEAN_HSP_U_P', 'H_CLEAN_LINEAR_SCORE', 'H_CLEAN_STAR_RATING', 'H_COMP_1_A_P', 'H_COMP_1_LINEAR_SCORE', 'H_COMP_1_SN_P', 'H_COMP_1_STAR_RATING', 'H_COMP_1_U_P', 'H_COMP_2_A_P', 'H_COMP_2_LINEAR_SCORE', 'H_COMP_2_SN_P', 'H_COMP_2_STAR_RATING', 'H_COMP_2_U_P', 'H_COMP_5_A_P', 'H_COMP_5_LINEAR_SCORE', 'H_COMP_5_SN_P', 'H_COMP_5_STAR_RATING', 'H_COMP_5_U_P', 'H_COMP_6_LINEAR_SCORE', 'H_COMP_6_N_P', 'H_COMP_6_STAR_RATING', 'H_COMP_6_Y_P', 'H_DISCH_HELP_N_P', 'H_DISCH_HELP_Y_P', 'H_DOCTOR_EXPLAIN_A_P', 'H_DOCTOR_EXPLAIN_SN_P', 'H_DOCTOR_EXPLAIN_U_P', 'H_DOCTOR_LISTEN_A_P', 'H_DOCTOR_LISTEN_SN_P', 'H_DOCTOR_LISTEN_U_P', 'H_DOCTOR_RESPECT_A_P', 'H_DOCTOR_RESPECT_SN_P', 'H_DOCTOR_RESPECT_U_P', 'H_HSP_RATING_0_6', 'H_HSP_RATING_7_8', 'H_HSP_RATING_9_10', 'H_HSP_RATING_LINEAR_SCORE', 'H_HSP_RATING_STAR_RATING', 'H_MED_FOR_A_P', 'H_MED_FOR_SN_

In [21]:
print(main_df.shape)
main_df.dropna(how='all', subset = cols, inplace=True)
print(main_df.shape)

labs = ['READM_30_HIP_KNEE', 'READM_30_COPD', 'MORT_30_STK', 'MORT_30_PN',
        'MORT_30_HF', 'MORT_30_COPD', 'MORT_30_AMI', 'COMP_HIP_KNEE', 'OP_22',
        'OP_23', 'OP_29', 'IMM_3', 'SEP_1', 'MORT_30_CABG',
        'READM_30_CABG', 'OP_8',
        'OP_10', 'OP_13', 
        'SAFE_USE_OF_OPIOIDS',
        'Hybrid_HWR',
        'Hybrid_HWM',
       ]

summary = {}
for col in labs:
    series = main_df[col]
    
    # Count NaNs and "Not Available"
    n_nans = series.isna().sum()
    n_notavail = (series == 'Not Available').sum()
    
    # Coerce to numeric (turns "Not Available" into NaN)
    numeric = pd.to_numeric(series, errors='coerce')
    
    # Count valid numeric entries
    n_numeric = numeric.notna().sum()
    
    summary[col] = {
        'min': numeric.min(skipna=True),
        'max': numeric.max(skipna=True),
        'mean': numeric.mean(skipna=True),
        'median': numeric.median(skipna=True),
        'numeric_count': n_numeric,
        'NaN_count': n_nans,
        "'Not Available'_count": n_notavail
    }

summary_df = pd.DataFrame(summary).T
print(summary_df)


(4792, 151)
(4792, 151)
                      min    max       mean  median  numeric_count  NaN_count  \
READM_30_HIP_KNEE     2.4    7.8   4.856062     4.8         1625.0        0.0   
READM_30_COPD        15.7   23.9  18.228746    18.1         2696.0        0.0   
MORT_30_STK           8.1   22.1  13.237732    13.1         2213.0        0.0   
MORT_30_PN            7.6   32.2  16.352087    16.2         3642.0        0.0   
MORT_30_HF            5.0   20.2  11.575403    11.5         3102.0        0.0   
MORT_30_COPD          4.5   16.5   8.874623     8.7         2656.0        0.0   
MORT_30_AMI           6.7   17.1  12.140266    12.1         1952.0        0.0   
COMP_HIP_KNEE         1.4    9.3   3.616133     3.5         1655.0        0.0   
OP_22                 0.0   23.0   1.662626     1.0         3960.0        4.0   
OP_23                 0.0  100.0  70.352027    74.0         1480.0      128.0   
OP_29                 0.0  100.0  92.725397    97.0         2957.0       28.0   
IMM_

In [22]:
print(main_df.shape)
main_df.dropna(how='all', subset = cols, inplace=True)
print(main_df.shape)

ls = ['READM_30_HIP_KNEE', 'READM_30_COPD', 'MORT_30_STK', 'MORT_30_PN',
      'MORT_30_HF', 'MORT_30_COPD', 'MORT_30_AMI', 'COMP_HIP_KNEE', 'OP_22',
      'OP_23', 'OP_29', 'IMM_3', 'SEP_1', 'MORT_30_CABG',
      'READM_30_CABG', 'OP_8',
      'OP_10', 'OP_13', 
      'SAFE_USE_OF_OPIOIDS',
      'Hybrid_HWM',
      'Hybrid_HWR', 
     ]

for col in list(main_df):
    if col != 'PROVIDER_ID':
        main_df[col] = pd.to_numeric(main_df[col], errors='coerce').astype(float)
        
for l in ls: 
    main_df[l] = main_df[l] * 0.01


(4792, 151)
(4792, 151)


# Notes
52 measures in 2026

### 1 measure was renamed:

READM_30_HOSP_WIDE: Rate of readmission after discharge from hospital (hospital-wide)

... renamed to ...

Hybrid_HWR: Hybrid Hospital-Wide All-Cause Readmission Rate

### Lost 2 measures:

PC_01: Percentage of mothers whose deliveries were scheduled too early (1-2 weeks early), when a scheduled
delivery wasn’t medically necessary

HCP_COVID_19: Percentage of employees that were COVID-19 vaccinated 

### Including 6 new measures for 2026:

Hybrid_HWM: Hybrid Hospital-Wide All-Cause Risk Standardized Mortality Rate

OQR_OAS_CAHPS:
O_COMP_1_LINEAR_SCORE
O_COMP_2_LINEAR_SCORE
O_COMP_3_LINEAR_SCORE
O_PATIENT_RATE_LINEAR_SCORE
O_PATIENT_REC_LINEAR_SCORE

### HCAHPS measure updates

H_INDI_STAR_RATING replaced with H_QUIET_LINEAR_SCORE and H_CLEAN_LINEAR_SCORE
H_GLOB_STAR_RATING replaced with H_HSP_RATING_LINEAR_SCORE and H_RECMND_LINEAR_SCORE


In [23]:
prvdrs = []
for p in main_df['PROVIDER_ID'].tolist():
    if 'F' in p:
        p = p[:-1]
        p = p + '666666'
    prvdrs.append(p)

main_df['PROVIDER_ID'] = prvdrs

main_df['PROVIDER_ID'] = pd.to_numeric(main_df['PROVIDER_ID'], errors='coerce')
main_df.sort_values(by=['PROVIDER_ID'], ascending = True, inplace = True)
main_df.to_csv(stars_dir + "2027/predictions_From_May_2026/data_for_2027_predictions_from_May2026.csv", 
               index=False)


In [24]:
for l in main_df.columns:
    if main_df[l].isna().all():
        print(l)

PSI_90_SAFETY_DEN
H_COMP_1_A_P
H_COMP_1_SN_P
H_COMP_1_U_P
H_COMP_1_STAR_RATING
H_NURSE_RESPECT_A_P
H_NURSE_RESPECT_SN_P
H_NURSE_RESPECT_U_P
H_NURSE_LISTEN_A_P
H_NURSE_LISTEN_SN_P
H_NURSE_LISTEN_U_P
H_NURSE_EXPLAIN_A_P
H_NURSE_EXPLAIN_SN_P
H_NURSE_EXPLAIN_U_P
H_COMP_2_A_P
H_COMP_2_SN_P
H_COMP_2_U_P
H_COMP_2_STAR_RATING
H_DOCTOR_RESPECT_A_P
H_DOCTOR_RESPECT_SN_P
H_DOCTOR_RESPECT_U_P
H_DOCTOR_LISTEN_A_P
H_DOCTOR_LISTEN_SN_P
H_DOCTOR_LISTEN_U_P
H_DOCTOR_EXPLAIN_A_P
H_DOCTOR_EXPLAIN_SN_P
H_DOCTOR_EXPLAIN_U_P
H_COMP_5_A_P
H_COMP_5_SN_P
H_COMP_5_U_P
H_COMP_5_STAR_RATING
H_MED_FOR_A_P
H_MED_FOR_SN_P
H_MED_FOR_U_P
H_SIDE_EFFECTS_A_P
H_SIDE_EFFECTS_SN_P
H_SIDE_EFFECTS_U_P
H_COMP_6_N_P
H_COMP_6_Y_P
H_COMP_6_STAR_RATING
H_DISCH_HELP_N_P
H_DISCH_HELP_Y_P
H_SYMPTOMS_N_P
H_SYMPTOMS_Y_P
H_CLEAN_HSP_A_P
H_CLEAN_HSP_SN_P
H_CLEAN_HSP_U_P
H_CLEAN_STAR_RATING
H_QUIET_HSP_A_P
H_QUIET_HSP_SN_P
H_QUIET_HSP_U_P
H_QUIET_STAR_RATING
H_HSP_RATING_0_6
H_HSP_RATING_7_8
H_HSP_RATING_9_10
H_HSP_RATING_STAR_RATING
H_R

In [25]:
main_df.head()

,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN,MORT_30_AMI,MORT_30_AMI_DEN,MORT_30_CABG,MORT_30_CABG_DEN,MORT_30_COPD,MORT_30_COPD_DEN,MORT_30_HF,MORT_30_HF_DEN,MORT_30_PN,MORT_30_PN_DEN,MORT_30_STK,MORT_30_STK_DEN,PSI_4_SURG_COMP,PSI_4_SURG_COMP_DEN,COMP_HIP_KNEE,COMP_HIP_KNEE_DEN,PSI_90_SAFETY,PSI_90_SAFETY_DEN,Hybrid_HWM,Hybrid_HWM_DEN,H_COMP_1_A_P,H_COMP_1_SN_P,H_COMP_1_U_P,H_COMP_1_LINEAR_SCORE,H_COMP_1_STAR_RATING,H_NURSE_RESPECT_A_P,H_NURSE_RESPECT_SN_P,H_NURSE_RESPECT_U_P,H_NURSE_LISTEN_A_P,H_NURSE_LISTEN_SN_P,H_NURSE_LISTEN_U_P,H_NURSE_EXPLAIN_A_P,H_NURSE_EXPLAIN_SN_P,H_NURSE_EXPLAIN_U_P,H_COMP_2_A_P,H_COMP_2_SN_P,H_COMP_2_U_P,H_COMP_2_LINEAR_SCORE,H_COMP_2_STAR_RATING,H_DOCTOR_RESPECT_A_P,H_DOCTOR_RESPECT_SN_P,H_DOCTOR_RESPECT_U_P,H_DOCTOR_LISTEN_A_P,H_DOCTOR_LISTEN_SN_P,H_DOCTOR_LISTEN_U_P,H_DOCTOR_EXPLAIN_A_P,H_DOCTOR_EXPLAIN_SN_P,H_DOCTOR_EXPLAIN_U_P,H_COMP_5_A_P,H_COMP_5_SN_P,H_COMP_5_U_P,H_COMP_5_LINEAR_SCORE,H_COMP_5_STAR_RATING,H_MED_FOR_A_P,H_MED_FOR_SN_P,H_MED_FOR_U_P,H_SIDE_EFFECTS_A_P,H_SIDE_EFFECTS_SN_P,H_SIDE_EFFECTS_U_P,H_COMP_6_N_P,H_COMP_6_Y_P,H_COMP_6_LINEAR_SCORE,H_COMP_6_STAR_RATING,H_DISCH_HELP_N_P,H_DISCH_HELP_Y_P,H_SYMPTOMS_N_P,H_SYMPTOMS_Y_P,H_CLEAN_HSP_A_P,H_CLEAN_HSP_SN_P,H_CLEAN_HSP_U_P,H_CLEAN_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_QUIET_HSP_A_P,H_QUIET_HSP_SN_P,H_QUIET_HSP_U_P,H_QUIET_LINEAR_SCORE,H_QUIET_STAR_RATING,H_HSP_RATING_0_6,H_HSP_RATING_7_8,H_HSP_RATING_9_10,H_HSP_RATING_LINEAR_SCORE,H_HSP_RATING_STAR_RATING,H_RECMND_DN,H_RECMND_DY,H_RECMND_PY,H_RECMND_LINEAR_SCORE,H_RECMND_STAR_RATING,H_STAR_RATING,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,EDAC_30_AMI,EDAC_30_AMI_DEN,EDAC_30_HF,EDAC_30_HF_DEN,EDAC_30_PN,EDAC_30_PN_DEN,OP_32,OP_32_DEN,OP_35_ADM,OP_35_ADM_DEN,OP_35_ED,OP_35_ED_DEN,OP_36,OP_36_DEN,READM_30_CABG,READM_30_CABG_DEN,READM_30_COPD,READM_30_COPD_DEN,READM_30_HIP_KNEE,READM_30_HIP_KNEE_DEN,Hybrid_HWR,Hybrid_HWR_DEN,HAI_1_DEN_PRED,HAI_1_DEN_VOL,HAI_1,HAI_2_DEN_PRED,HAI_2_DEN_VOL,HAI_2,HAI_3_DEN_PRED,HAI_3_DEN_VOL,HAI_3,HAI_4_DEN_PRED,HAI_4_DEN_VOL,HAI_4,HAI_5_DEN_PRED,HAI_5_DEN_VOL,HAI_5,HAI_6_DEN_PRED,HAI_6_DEN_VOL,HAI_6,OP_13,OP_8,OP_10
0,10001,0.93,4625.0,217.0,395.0,0.03,62163.0,NaN,NaN,0.62,39.0,0.65,141.0,0.14,4583.0,0.114,270.0,0.030,144.0,0.094,112.0,0.102,583.0,0.184,517.0,0.135,395.0,203.00,118.0,0.032,27.0,0.95,NaN,0.045,1835.0,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,78.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.0,NaN,NaN,NaN,NaN,87.0,NaN,NaN,NaN,NaN,89.0,NaN,NaN,NaN,NaN,91.0,NaN,NaN,98.0,92.0,97.0,94.0,94.0,-15.6,273.0,-1.1,652.0,17.4,507.0,12.7,234.0,10.0,302.0,5.1,302.0,1.1,651.0,0.101,137.0,0.180,122.0,0.048,25.0,0.151,2824.0,10.399,9871.0,0.288,25.693,18164.0,0.195,6.830,238.0,0.293,0.871,91.0,NaN,9.613,114863.0,0.416,71.953,113805.0,0.459,0.038,0.308,0.053
1,10005,0.59,2856.0,141.0,1165.0,0.03,60663.0,NaN,NaN,0.98,301.0,0.73,293.0,0.15,1859.0,NaN,NaN,NaN,NaN,0.089,126.0,0.141,158.0,0.212,285.0,0.129,89.0,184.79,27.0,0.030,104.0,0.97,NaN,0.046,698.0,NaN,NaN,NaN,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,85.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.0,NaN,NaN,NaN,NaN,86.0,NaN,NaN,NaN,NaN,86.0,NaN,NaN,NaN,NaN,84.0,NaN,NaN,98.0,96.0,98.0,94.0,93.0,NaN,NaN,12.2,164.0,-17.2,292.0,13.0,917.0,9.3,84.0,4.8,84.0,0.9,389.0,NaN,NaN,0.171,132.0,0.042,81.0,0.133,986.0,2.952,4837.0,1.694,2.904,6302.0,1.722,2.175,85.0,0.920,0.517,49.0,NaN,1.295,37659.0,1.544,9.407,35960.0,0.425,0.033,0.422,0.128
2,10006,0.64,2565.0,144.0,320.0,0.01,46884.0,0.82,17.0,0.84,67.0,0.61,137.0,0.15,4350.0,0.145,266.0,0.054,79.0,0.087,160.0,0.125,413.0,0.196,659.0,0.124,258.0,236.12,91.0,0.047,49.0,1.14,NaN,0.052,1583.0,NaN,NaN,NaN,89.0,NaN,NaN,NaN,NaN,NaN,N

In [26]:
raw_data = pd.read_csv(stars_dir + "2027/predictions_From_May_2026/data_for_2027_predictions_from_May2026.csv")
raw_data.head()

df_dupes = raw_data[raw_data.duplicated('PROVIDER_ID', keep=False)]

print(df_dupes.shape)
df_dupes.head()

(0, 151)


,PROVIDER_ID,IMM_3,IMM_3_DEN,OP_18B,OP_18B_DEN,OP_22,OP_22_DEN,OP_23,OP_23_DEN,OP_29,OP_29_DEN,SEP_1,SEP_1_DEN,SAFE_USE_OF_OPIOIDS,SAFE_USE_OF_OPIOIDS_DEN,MORT_30_AMI,MORT_30_AMI_DEN,MORT_30_CABG,MORT_30_CABG_DEN,MORT_30_COPD,MORT_30_COPD_DEN,MORT_30_HF,MORT_30_HF_DEN,MORT_30_PN,MORT_30_PN_DEN,MORT_30_STK,MORT_30_STK_DEN,PSI_4_SURG_COMP,PSI_4_SURG_COMP_DEN,COMP_HIP_KNEE,COMP_HIP_KNEE_DEN,PSI_90_SAFETY,PSI_90_SAFETY_DEN,Hybrid_HWM,Hybrid_HWM_DEN,H_COMP_1_A_P,H_COMP_1_SN_P,H_COMP_1_U_P,H_COMP_1_LINEAR_SCORE,H_COMP_1_STAR_RATING,H_NURSE_RESPECT_A_P,H_NURSE_RESPECT_SN_P,H_NURSE_RESPECT_U_P,H_NURSE_LISTEN_A_P,H_NURSE_LISTEN_SN_P,H_NURSE_LISTEN_U_P,H_NURSE_EXPLAIN_A_P,H_NURSE_EXPLAIN_SN_P,H_NURSE_EXPLAIN_U_P,H_COMP_2_A_P,H_COMP_2_SN_P,H_COMP_2_U_P,H_COMP_2_LINEAR_SCORE,H_COMP_2_STAR_RATING,H_DOCTOR_RESPECT_A_P,H_DOCTOR_RESPECT_SN_P,H_DOCTOR_RESPECT_U_P,H_DOCTOR_LISTEN_A_P,H_DOCTOR_LISTEN_SN_P,H_DOCTOR_LISTEN_U_P,H_DOCTOR_EXPLAIN_A_P,H_DOCTOR_EXPLAIN_SN_P,H_DOCTOR_EXPLAIN_U_P,H_COMP_5_A_P,H_COMP_5_SN_P,H_COMP_5_U_P,H_COMP_5_LINEAR_SCORE,H_COMP_5_STAR_RATING,H_MED_FOR_A_P,H_MED_FOR_SN_P,H_MED_FOR_U_P,H_SIDE_EFFECTS_A_P,H_SIDE_EFFECTS_SN_P,H_SIDE_EFFECTS_U_P,H_COMP_6_N_P,H_COMP_6_Y_P,H_COMP_6_LINEAR_SCORE,H_COMP_6_STAR_RATING,H_DISCH_HELP_N_P,H_DISCH_HELP_Y_P,H_SYMPTOMS_N_P,H_SYMPTOMS_Y_P,H_CLEAN_HSP_A_P,H_CLEAN_HSP_SN_P,H_CLEAN_HSP_U_P,H_CLEAN_LINEAR_SCORE,H_CLEAN_STAR_RATING,H_QUIET_HSP_A_P,H_QUIET_HSP_SN_P,H_QUIET_HSP_U_P,H_QUIET_LINEAR_SCORE,H_QUIET_STAR_RATING,H_HSP_RATING_0_6,H_HSP_RATING_7_8,H_HSP_RATING_9_10,H_HSP_RATING_LINEAR_SCORE,H_HSP_RATING_STAR_RATING,H_RECMND_DN,H_RECMND_DY,H_RECMND_PY,H_RECMND_LINEAR_SCORE,H_RECMND_STAR_RATING,H_STAR_RATING,O-COMP-1,O-COMP-2,O-COMP-3,O-PATIENT-RATE,O-PATIENT-REC,EDAC_30_AMI,EDAC_30_AMI_DEN,EDAC_30_HF,EDAC_30_HF_DEN,EDAC_30_PN,EDAC_30_PN_DEN,OP_32,OP_32_DEN,OP_35_ADM,OP_35_ADM_DEN,OP_35_ED,OP_35_ED_DEN,OP_36,OP_36_DEN,READM_30_CABG,READM_30_CABG_DEN,READM_30_COPD,READM_30_COPD_DEN,READM_30_HIP_KNEE,READM_30_HIP_KNEE_DEN,Hybrid_HWR,Hybrid_HWR_DEN,HAI_1_DEN_PRED,HAI_1_DEN_VOL,HAI_1,HAI_2_DEN_PRED,HAI_2_DEN_VOL,HAI_2,HAI_3_DEN_PRED,HAI_3_DEN_VOL,HAI_3,HAI_4_DEN_PRED,HAI_4_DEN_VOL,HAI_4,HAI_5_DEN_PRED,HAI_5_DEN_VOL,HAI_5,HAI_6_DEN_PRED,HAI_6_DEN_VOL,HAI_6,OP_13,OP_8,OP_10


In [27]:
measures = [
    'Hybrid_HWR',
      'Hybrid_HWM',
      'O-COMP-1',
      'O-COMP-2',
      'O-COMP-3',
      'O-PATIENT-RATE',
      'O-PATIENT-REC',
      'H_COMP_1_LINEAR_SCORE',
      'H_COMP_2_LINEAR_SCORE', 
      #'H_COMP_3_LINEAR_SCORE', 
      'H_COMP_5_LINEAR_SCORE', 
      'H_COMP_6_LINEAR_SCORE', 
      #'H_COMP_7_LINEAR_SCORE', 
      'H_CLEAN_LINEAR_SCORE',  
      'H_QUIET_LINEAR_SCORE', 
      'H_RECMND_LINEAR_SCORE', 
      'H_HSP_RATING_LINEAR_SCORE',
]

for m in measures:
    ls = raw_data[m]
    print(m)
    print('   ', np.nanmin(ls), np.nanmin(ls), np.nanmedian(ls), np.nanmean(ls))

Hybrid_HWR
    0.1169999999999999 0.1169999999999999 0.149 0.14980052244122535
Hybrid_HWM
    0.018 0.018 0.042 0.04218108651911468
O-COMP-1
    80.0 80.0 98.0 98.17221693625119
O-COMP-2
    80.0 80.0 95.0 95.28068506184586
O-COMP-3
    87.0 87.0 98.0 97.5359974627339
O-PATIENT-RATE
    69.0 69.0 94.0 94.04440215667618
O-PATIENT-REC
    64.0 64.0 93.0 92.73897875039644
H_COMP_1_LINEAR_SCORE
    74.0 74.0 91.0 91.24338790931989
H_COMP_2_LINEAR_SCORE
    74.0 74.0 91.0 90.67443324937028
H_COMP_5_LINEAR_SCORE
    57.0 57.0 77.0 76.70749370277078
H_COMP_6_LINEAR_SCORE
    65.0 65.0 86.0 85.66404282115869
H_CLEAN_LINEAR_SCORE
    69.0 69.0 87.0 86.56202770780857
H_QUIET_LINEAR_SCORE
    55.0 55.0 82.0 81.45560453400503
H_RECMND_LINEAR_SCORE
    60.0 60.0 87.0 86.96253148614609
H_HSP_RATING_LINEAR_SCORE
    66.0 66.0 88.0 87.8838161209068
